# Aula 13C — Model Routing, Orchestration e Utility

**Da comparação entre modelos à engenharia de sistemas compostos de IA**

Nesta extensão da Aula 13, a unidade de avaliação deixa de ser apenas o modelo e passa a ser o **sistema**. Vamos combinar qualidade, custo e latência para decidir quando usar um modelo econômico, quando escalar para um modelo mais forte e quando uma segunda avaliação faz sentido.

> Pergunta central: **qual combinação de modelos, regras e mecanismos de avaliação entrega valor suficiente com custo, latência e risco aceitáveis?**


## 1. Model Selection → Routing → Orchestration → Compound AI Systems

```text
entrada
  ↓
router
  ↓
┌────────────────────────────────┐
│ tarefa simples → modelo barato │
│ baixa confiança → modelo forte │
│ tarefa crítica → modelo+crítico│
└────────────────────────────────┘
  ↓
quality gate
  ↓
resposta
```

Três estratégias úteis: **single**, **cascade** e **critique**. O `single` usa um modelo; o `cascade` começa barato e escala apenas quando necessário; o `critique` adiciona uma avaliação independente antes da resposta final.


## 2. Estudo de caso — GitHub Project HydraFusion

O Project HydraFusion é um exemplo contemporâneo de **multi-model orchestration** em programação assistida por IA. O ponto pedagógico para o TIL não é declarar um vencedor, mas observar a mudança de problema: o objetivo passa a ser otimizar uma arquitetura inteira.

Os resultados divulgados pelo GitHub mostraram reduções relevantes de custo, enquanto a superioridade em qualidade não foi uniforme em todos os benchmarks. Isso ilustra por que qualidade, custo e latência devem ser avaliados conjuntamente.

Leituras: [GitHub Blog — Project HydraFusion](https://github.blog/ai-and-ml/github-copilot/project-hydrafusion-frontier-quality-via-multi-model-orchestration/) e [VentureBeat — análise crítica](https://venturebeat.com/orchestration/githubs-hydrafusion-cuts-ai-coding-costs-in-every-benchmark-it-only-matches-quality-in-one).

> Benchmarks dependem de dataset, configuração, preços e critérios. Use-os como evidência contextual, não como verdade universal.


## 3. Utility: tornando prioridades explícitas

Uma função didática de utilidade pode ser escrita como:

`U = wq·Q − wc·C − wl·L`

onde `Q` é qualidade, `C` é custo, `L` é latência e os pesos representam prioridades do negócio. Não é uma métrica universal: é uma forma transparente de declarar trade-offs.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

systems = pd.DataFrame({
    'system': ['single_small', 'single_frontier', 'cascade', 'critique'],
    'quality': [0.82, 0.95, 0.93, 0.96],
    'cost': [0.20, 1.00, 0.42, 1.55],
    'latency': [0.25, 0.80, 0.55, 1.20],
})

df = systems.copy()
for col in ['quality', 'cost', 'latency']:
    lo, hi = df[col].min(), df[col].max()
    df[col + '_norm'] = (df[col] - lo) / (hi - lo)

def utility_table(w_quality=0.60, w_cost=0.25, w_latency=0.15):
    out = df.copy()
    out['utility'] = (
        w_quality * out['quality_norm']
        - w_cost * out['cost_norm']
        - w_latency * out['latency_norm']
    )
    return out.sort_values('utility', ascending=False)

utility_table().round(3)


## 4. Desafio — o vencedor muda com o contexto?

Experimente três cenários: qualidade primeiro `(0.80, 0.10, 0.10)`, operação sensível a custo `(0.50, 0.40, 0.10)` e resposta sensível a latência `(0.50, 0.10, 0.40)`. Observe se a mesma arquitetura vence sempre.


In [ ]:
scenarios = {
    'quality_first': (0.80, 0.10, 0.10),
    'cost_sensitive': (0.50, 0.40, 0.10),
    'latency_sensitive': (0.50, 0.10, 0.40),
}

rows = []
for name, weights in scenarios.items():
    ranking = utility_table(*weights)
    winner = ranking.iloc[0]
    rows.append({'scenario': name, 'winner': winner['system'], 'utility': winner['utility']})

pd.DataFrame(rows).round(3)


## 5. Simulando um cascade e o quality gate

No `cascade`, um modelo econômico atende primeiro. Apenas uma fração das requisições é escalada. A **taxa de escalonamento** é uma das métricas centrais do sistema.

Um `quality gate` pode usar confiança, regras determinísticas, validação de formato, testes, evidência recuperada, um modelo avaliador ou revisão humana.


In [ ]:
def cascade_metrics(escalation_rate, cheap_quality=0.82, premium_quality=0.95,
                    cheap_cost=0.20, premium_cost=1.00,
                    cheap_latency=0.25, premium_latency=0.80):
    r = escalation_rate
    return {
        'escalation_rate': r,
        'quality': (1-r)*cheap_quality + r*premium_quality,
        'cost': cheap_cost + r*premium_cost,
        'latency': cheap_latency + r*premium_latency,
    }

rates = np.arange(0, 1.01, 0.10)
cascade_df = pd.DataFrame([cascade_metrics(r) for r in rates])
display(cascade_df.round(3))

plt.figure(figsize=(7, 4))
plt.plot(cascade_df['cost'], cascade_df['quality'], marker='o')
plt.xlabel('Custo relativo')
plt.ylabel('Qualidade')
plt.title('Cascade — trade-off entre custo e qualidade')
plt.grid(True, alpha=0.3)
plt.show()


## 6. Métricas do sistema e exercício final

Além de F1 ou task success, acompanhe: custo por requisição, latência p50/p95/p99, taxa de escalonamento, aprovação do gate, falso aceite, falso bloqueio, retries, timeout, taxa de revisão humana e aceitação da resposta.

### Exercício
Projete um sistema TIL com três níveis: **TF-IDF + Naive Bayes** como baseline, **Transformer** para baixa confiança e **LLM ou revisão humana** apenas para casos críticos. Defina regra de roteamento, quality gate, custo por camada, métricas e condição de escalonamento.

### Síntese
```text
Qual modelo tem a maior métrica?
            ↓
Qual sistema entrega valor suficiente
com custo, latência e risco aceitáveis?
```
